CELDA 1 — Instalación de librerías

In [ ]:
!pip install tensorflow scikit-learn pymongo dnspython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.7 MB/s eta 0:00:00


CELDA 2 — Librerías

In [ ]:
import numpy as np
import pandas as pd
from pymongo import MongoClient
from datetime import datetime, timezone
from google.colab import userdata

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CELDA 3 — Conexión a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 60          # días de contexto, tal como especifica el enunciado
EPOCAS = 100
BATCH_SIZE = 32
HORIZONTES = [7, 14, 30, 60]

MONGO_URI = userdata.get("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["ernesto_investing_ai"]

coleccion_precios = db["precios_ohlcv"]
coleccion_predicciones = db["predicciones"]
coleccion_metricas = db["metricas_modelos"]

try:
    client.admin.command("ping")
    print("✅ Conexión a MongoDB Atlas exitosa.")
except Exception as e:
    print("❌ Error de conexión:", e)

✅ Conexión a MongoDB Atlas exitosa.


CELDA 4 — Función: leer de Mongo y preparar secuencias (target continuo)

In [ ]:
def leer_precios_de_mongo(ticker):
    doc = coleccion_precios.find_one({"ticker": ticker})
    if doc is None:
        raise ValueError(f"No hay datos en Mongo para {ticker}. Corre primero el Notebook 1.")

    df = pd.DataFrame({
        "Close": doc["close"],
    }, index=pd.to_datetime(doc["fechas"]))

    df.dropna(inplace=True)
    return df


def preparar_secuencias_regresion(df, ventana=VENTANA):
    """
    Ventanas deslizantes de 'ventana' días para predecir el precio del día siguiente.
    """
    scaler = MinMaxScaler()
    precios_norm = scaler.fit_transform(df[["Close"]])

    X, y = [], []
    for i in range(ventana, len(precios_norm)):
        X.append(precios_norm[i - ventana:i, 0])
        y.append(precios_norm[i, 0])

    X = np.array(X).reshape(-1, ventana, 1)
    y = np.array(y)

    corte = int(len(X) * 0.8)
    X_train, X_test = X[:corte], X[corte:]
    y_train, y_test = y[:corte], y[corte:]

    return X_train, X_test, y_train, y_test, precios_norm, scaler

CELDA 5 — Arquitectura del LSTM Regressor

In [ ]:
def construir_lstm_regressor(input_shape):
    modelo = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear")
    ])
    modelo.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return modelo

CELDA 6 — Función: entrenar, evaluar y proyectar horizonte futuro

In [ ]:
def entrenar_lstm_regressor(ticker):
    df = leer_precios_de_mongo(ticker)
    X_train, X_test, y_train, y_test, precios_norm, scaler = preparar_secuencias_regresion(df)

    modelo = construir_lstm_regressor((X_train.shape[1], 1))
    modelo.fit(
        X_train, y_train,
        epochs=EPOCAS,
        batch_size=BATCH_SIZE,
        verbose=0,
        validation_split=0.1
    )

    # Evaluación en test
    pred_test_norm = modelo.predict(X_test, verbose=0).flatten()
    pred_test = scaler.inverse_transform(pred_test_norm.reshape(-1, 1)).flatten()
    real_test = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse_usd = float(np.sqrt(mean_squared_error(real_test, pred_test)))
    mae_usd = float(mean_absolute_error(real_test, pred_test))
    r2 = float(r2_score(real_test, pred_test))
    precio_medio = float(np.mean(real_test))
    rmse_pct = round((rmse_usd / precio_medio) * 100, 4)

    # Proyección futura: predicción recursiva día a día
    secuencia_actual = precios_norm[-VENTANA:].reshape(1, VENTANA, 1)
    predicciones_futuras_norm = []

    for _ in range(max(HORIZONTES)):
        siguiente = modelo.predict(secuencia_actual, verbose=0)[0][0]
        predicciones_futuras_norm.append(siguiente)
        secuencia_actual = np.append(secuencia_actual[:, 1:, :], [[[siguiente]]], axis=1)

    predicciones_futuras = scaler.inverse_transform(
        np.array(predicciones_futuras_norm).reshape(-1, 1)
    ).flatten()

    # Banda de confianza simple: ±1.96 * RMSE (aprox. 95%)
    banda_superior = predicciones_futuras + 1.96 * rmse_usd
    banda_inferior = predicciones_futuras - 1.96 * rmse_usd

    return {
        "rmse_usd": round(rmse_usd, 4),
        "rmse_pct": rmse_pct,
        "mae_usd": round(mae_usd, 4),
        "r2": round(r2, 4),
        "precios_reales_test": [round(float(x), 4) for x in real_test],
        "precios_predichos_test": [round(float(x), 4) for x in pred_test],
        "fechas_test": df.index[-len(real_test):].strftime("%Y-%m-%d").tolist(),
        "prediccion_futura": {
            str(h): {
                "precio": round(float(predicciones_futuras[h - 1]), 4),
                "banda_superior": round(float(banda_superior[h - 1]), 4),
                "banda_inferior": round(float(banda_inferior[h - 1]), 4)
            } for h in HORIZONTES
        },
        "serie_futura_completa": [round(float(x), 4) for x in predicciones_futuras]
    }

CELDA 7 — Función: guardar resultados en Mongo

In [ ]:
def guardar_resultados_lstm_reg(ticker, resultado):
    ahora = datetime.now(timezone.utc)

    documento_prediccion = {
        "ticker": ticker,
        "modelo": "LSTM_Regressor",
        "fechas_test": resultado["fechas_test"],
        "precios_reales_test": resultado["precios_reales_test"],
        "precios_predichos_test": resultado["precios_predichos_test"],
        "prediccion_futura": resultado["prediccion_futura"],
        "serie_futura_completa": resultado["serie_futura_completa"],
        "fecha_actualizacion": ahora
    }

    documento_metricas = {
        "ticker": ticker,
        "modelo": "LSTM_Regressor",
        "rmse_usd": resultado["rmse_usd"],
        "rmse_pct": resultado["rmse_pct"],
        "mae_usd": resultado["mae_usd"],
        "r2": resultado["r2"],
        "hiperparametros": {
            "ventana": VENTANA,
            "epocas": EPOCAS,
            "batch_size": BATCH_SIZE
        },
        "fecha_actualizacion": ahora
    }

    coleccion_predicciones.update_one(
        {"ticker": ticker, "modelo": "LSTM_Regressor"},
        {"$set": documento_prediccion},
        upsert=True
    )

    coleccion_metricas.update_one(
        {"ticker": ticker, "modelo": "LSTM_Regressor"},
        {"$set": documento_metricas},
        upsert=True
    )

CELDA 8 — Ejecución: loop sobre los 5 tickers

In [ ]:
resultados_regressor = {}

for ticker in TICKERS:
    print(f"Entrenando LSTM Regressor para {ticker}...")
    try:
        resultado = entrenar_lstm_regressor(ticker)
        guardar_resultados_lstm_reg(ticker, resultado)
        resultados_regressor[ticker] = resultado

        print(f"  RMSE: ${resultado['rmse_usd']} ({resultado['rmse_pct']}%)")
        print(f"  MAE: ${resultado['mae_usd']} | R²: {resultado['r2']}")
        print(f"  Predicción a 30 días: ${resultado['prediccion_futura']['30']['precio']}")
        print(f"  ✅ Guardado en MongoDB.\n")

    except Exception as e:
        print(f"  ❌ Error con {ticker}: {e}\n")

Entrenando LSTM Regressor para FSM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  RMSE: $0.5935 (6.1354%)
  MAE: $0.4827 | R²: 0.3963
  Predicción a 30 días: $8.9703
  ✅ Guardado en MongoDB.

Entrenando LSTM Regressor para VOLCABC1.LM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  RMSE: $0.0609 (8.2422%)
  MAE: $0.0494 | R²: 0.3701
  Predicción a 30 días: $0.4845
  ✅ Guardado en MongoDB.

Entrenando LSTM Regressor para ABX.TO...


...

  RMSE: $3.4177 (6.0786%)
  MAE: $2.7957 | R²: -0.3918
  Predicción a 30 días: $52.0624
  ✅ Guardado en MongoDB.

Entrenando LSTM Regressor para BVN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  RMSE: $2.2095 (6.5962%)
  MAE: $1.7572 | R²: 0.14
  Predicción a 30 días: $31.069
  ✅ Guardado en MongoDB.

Entrenando LSTM Regressor para BHP...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  RMSE: $6.8604 (8.5311%)
  MAE: $5.4868 | R²: 0.0575
  Predicción a 30 días: $337.9565
  ✅ Guardado en MongoDB.



CELDA 9 — Verificación final

In [ ]:
print("Verificación — RMSE y proyección a 30 días por ticker:\n")

for doc in coleccion_metricas.find({"modelo": "LSTM_Regressor"}, {"_id": 0}):
    pred = coleccion_predicciones.find_one(
        {"ticker": doc["ticker"], "modelo": "LSTM_Regressor"},
        {"prediccion_futura.30": 1, "_id": 0}
    )
    print(f"{doc['ticker']}: RMSE=${doc['rmse_usd']} ({doc['rmse_pct']}%) | "
          f"R²={doc['r2']} | Predicción 30d=${pred['prediccion_futura']['30']['precio']}")

Verificación — RMSE y proyección a 30 días por ticker:

FSM: RMSE=$0.5935 (6.1354%) | R²=0.3963 | Predicción 30d=$8.9703
VOLCABC1.LM: RMSE=$0.0609 (8.2422%) | R²=0.3701 | Predicción 30d=$0.4845
ABX.TO: RMSE=$3.4177 (6.0786%) | R²=-0.3918 | Predicción 30d=$52.0624
BVN: RMSE=$2.2095 (6.5962%) | R²=0.14 | Predicción 30d=$31.069
BHP: RMSE=$6.8604 (8.5311%) | R²=0.0575 | Predicción 30d=$337.9565
